# FLARE ancestry-switch QC (GQ / DP, old vs new)

Two parts, same CLI (`scripts/flare_switch_qc.py`):

**Part 1 — GQ / DP / RNC** on an annotated FLARE backbone (`AnnotateFlareGqDp`: `GQ`, `DP`, `AN1`, `AN2`, `GT`, `RNC`):

1. Find **ancestry switches** (haplotype `AN1`/`AN2` changes)
2. Flag short **flickers** (A→B→A within `--flicker-max-bp`, default 50 kb)
3. Compare **GQ / DP** at switches vs background, overall and by genotype class
4. Record GQ/DP of the **nearest het** (most switches land on GLnexus `0/0`)
5. Split GLnexus **`RNC`**: `..` is a called placeholder; `I` is incomplete gVCF still emitted as `0/0`

**Part 2 — old vs new FLARE** on the same samples and region. The new `FlareByPopulation` `*.anc.vcf.gz` only needs `AN1`/`AN2`. Set `NEW_VCF` (and `NEW_MODELS` if you have `*.models.tsv`). The comparison re-scans both VCFs on the **intersection** of samples so dropping controls / EUR-EAS does not inflate a rate difference.

In a 1 Mb window, tract-length ECDFs are mostly censored. The fair metric is **switches per hap per Mb**, compared with T/100 (1 cM/Mb). Full chr1 is where median tract length becomes meaningful.

`PropagateAnnotations.wdl` drops `GT`, so the genotype split in part 1 will be empty.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py", "flare_switch_qc.py")
from workspace_paths import data_root

ROOT = data_root()
OUT = ROOT / "flare_switch_qc"
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("OUT:", OUT)
print("scripts:", SCRIPTS)

## Config

Defaults are the 1 Mb chr22 pair already used for this investigation (old `AnnotateFlareGqDp` backbone + `FlareByPopulation` merged anc VCF). Env vars still override.

`FORCE_RELOCALIZE` / `FORCE_COMPARE` default **on** so a kernel re-run actually re-scans after a CLI change. Flip them to `False` once outputs look right.

For chr1: point `NEW_VCF` / `NEW_MODELS` at the train-chrom outputs and leave `COMPARE_REGION` empty.

In [ ]:
import os
from pathlib import Path

# 1 Mb chr22 investigation (AnnotateFlareGqDp + FlareByPopulation smoke test)
_DEFAULT_OLD_VCF = (
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/submissions/"
    "f8ab43b1-5506-4182-906b-3c6567033f1c/AnnotateFlareGqDp/"
    "8a5bd1e2-2be1-4d97-865d-2d32054a8478/call-AnnotateRegion/cacheCopy/"
    "flare_gq_dp.flare.gq_dp.vcf.gz"
)
_DEFAULT_NEW_VCF = (
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/submissions/"
    "00caa140-b8bf-48e1-8351-d58e6ae9a5cf/FlareByPopulation/"
    "ff596b36-5530-452a-baf8-1cf2ae2b07eb/call-MergePopulationFlare/"
    "aou_lr_phase2_v1.chr22.anc.vcf.gz"
)
_DEFAULT_NEW_MODELS = (
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/submissions/"
    "00caa140-b8bf-48e1-8351-d58e6ae9a5cf/FlareByPopulation/"
    "ff596b36-5530-452a-baf8-1cf2ae2b07eb/call-MergePopulationFlare/"
    "aou_lr_phase2_v1.chr22.models.tsv"
)

ANNOTATED_VCF = os.environ.get("ANNOTATED_VCF", _DEFAULT_OLD_VCF).strip()
assert ANNOTATED_VCF, "Set ANNOTATED_VCF env var or edit this cell"

# Optional: one sample ID per line. Empty → all VCF samples.
SAMPLES = os.environ.get("ANALYSIS_SAMPLES", "")

PREFIX = os.environ.get("SWITCH_QC_PREFIX", "chr22")
GQ_THRESHOLD = int(os.environ.get("GQ_THRESHOLD", "20"))
DP_THRESHOLD = int(os.environ.get("DP_THRESHOLD", "5"))
FLICKER_MAX_BP = int(os.environ.get("FLICKER_MAX_BP", "50000"))
BG_KEEP_EVERY = int(os.environ.get("BG_KEEP_EVERY", "50"))

# Smoke test: set e.g. 100000 to scan only the first N sites
MAX_SITES = os.environ.get("MAX_SITES")
MAX_SITES = int(MAX_SITES) if MAX_SITES else None

FORCE_RELOCALIZE = True
FORCE_COMPARE = True

# Optional bcftools -r for part 1 and/or the old-vs-new comparison.
REGION = os.environ.get("SWITCH_QC_REGION", "").strip()
COMPARE_REGION = os.environ.get("COMPARE_REGION", REGION).strip()

# FlareByPopulation merged anc VCF (AN1/AN2 only is fine) and models.tsv.
NEW_VCF = os.environ.get("NEW_FLARE_VCF", _DEFAULT_NEW_VCF).strip()
NEW_MODELS = os.environ.get("NEW_FLARE_MODELS", _DEFAULT_NEW_MODELS).strip()
OLD_FLARE_T = float(os.environ.get("OLD_FLARE_T", "120.3"))
COVARIATES = os.environ.get(
    "FLARE_COVARIATES",
    str(ROOT / "covariates.source_rebuilt.csv.gz") if (ROOT / "covariates.source_rebuilt.csv.gz").is_file() else "",
)

RUN_DIR = OUT / PREFIX
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_VCF = RUN_DIR / Path(ANNOTATED_VCF).name

print("ANNOTATED_VCF:", ANNOTATED_VCF)
print("NEW_VCF:", NEW_VCF or "(skip old-vs-new)")
print("NEW_MODELS:", NEW_MODELS or "(none)")
print("COVARIATES:", COVARIATES or "(none)")
print("SAMPLES:", SAMPLES or "(all)")
print("REGION:", REGION or "(VCF as-is)")
print("COMPARE_REGION:", COMPARE_REGION or "(VCF as-is)")
print("OLD_FLARE_T:", OLD_FLARE_T)
print("RUN_DIR:", RUN_DIR)
print("LOCAL_VCF:", LOCAL_VCF)
print(f"thresholds: GQ<{GQ_THRESHOLD}, DP<{DP_THRESHOLD}, flicker≤{FLICKER_MAX_BP} bp")
print("FORCE_RELOCALIZE:", FORCE_RELOCALIZE)
print("FORCE_COMPARE:", FORCE_COMPARE)

## Localize VCF (if GCS) and run switch scan

In [ ]:
import shutil
import subprocess
from pathlib import Path

assert shutil.which("bcftools"), "bcftools required on PATH"

src = ANNOTATED_VCF
if src.startswith("gs://"):
    def _hdr_has(path: Path, tag: str) -> bool:
        if not path.is_file():
            return False
        h = subprocess.check_output(["bcftools", "view", "-h", str(path)], text=True)
        return f"ID={tag}," in h

    stale = LOCAL_VCF.is_file() and not _hdr_has(LOCAL_VCF, "RNC")
    if FORCE_RELOCALIZE or not LOCAL_VCF.is_file() or stale:
        if stale:
            print(
                f"local VCF {LOCAL_VCF} has no FORMAT/RNC (previous GQ/DP-only copy); "
                "re-downloading from GCS"
            )
        print(f"gsutil cp {src} {LOCAL_VCF}")
        subprocess.check_call(["gsutil", "-m", "cp", src, str(LOCAL_VCF)])
        tbi = src + ".tbi"
        try:
            subprocess.check_call(["gsutil", "-m", "cp", tbi, str(LOCAL_VCF) + ".tbi"])
        except subprocess.CalledProcessError:
            print("no .tbi alongside VCF; continuing")
    else:
        print(f"using cached {LOCAL_VCF}")
    vcf_path = LOCAL_VCF
else:
    vcf_path = Path(src)
    assert vcf_path.is_file(), vcf_path

print("scanning:", vcf_path)

# AN1/AN2 required. GQ/DP enable part 1 histograms.
hdr = subprocess.check_output(["bcftools", "view", "-h", str(vcf_path)], text=True)
for tag in ("AN1", "AN2"):
    assert f"ID={tag}," in hdr, f"missing FORMAT {tag} in {vcf_path}"
print("FORMAT AN1/AN2 OK")
for tag, msg in (
    ("GQ", "quality histograms empty"),
    ("DP", "quality histograms empty"),
    ("GT", "het vs hom_ref split will be empty. Use AnnotateFlareGqDp output"),
    ("RNC", "re-run AnnotateFlareGqDp with format_tags=GQ,DP,RNC"),
):
    if f"ID={tag}," in hdr:
        print(f"FORMAT/{tag} OK")
    else:
        print(f"WARNING: no FORMAT/{tag}; {msg}")

cmd = [
    sys.executable,
    str(SCRIPTS / "flare_switch_qc.py"),
    "--vcf",
    str(vcf_path),
    "--out-dir",
    str(RUN_DIR),
    "--gq-threshold",
    str(GQ_THRESHOLD),
    "--dp-threshold",
    str(DP_THRESHOLD),
    "--flicker-max-bp",
    str(FLICKER_MAX_BP),
    "--bg-keep-every",
    str(BG_KEEP_EVERY),
]
if REGION:
    cmd.extend(["--region", REGION])
if SAMPLES:
    samp = Path(SAMPLES)
    if str(SAMPLES).startswith("gs://"):
        local_samp = RUN_DIR / "samples.txt"
        subprocess.check_call(["gsutil", "cp", SAMPLES, str(local_samp)])
        samp = local_samp
    cmd.extend(["--samples", str(samp)])
if MAX_SITES is not None:
    cmd.extend(["--max-sites", str(MAX_SITES)])

print(" ".join(cmd))
subprocess.check_call(cmd)

## Load results

In [ ]:
import json
import pandas as pd

summary = json.loads((RUN_DIR / "summary.json").read_text())
switches = pd.read_csv(RUN_DIR / "switches.tsv.gz", sep="\t", compression="gzip")

print("sites scanned:", summary["sites_scanned"])
print("switches:", summary["n_switches"], "flickers:", summary["n_flicker_switches"])
tr = summary.get("tracts") or {}
if tr:
    print("\nTracts / rate:")
    for k in (
        "span_mb",
        "switches_per_hap_per_mb",
        "flicker_frac",
        "frac_haps_with_switch",
        "implied_t_gen",
        "median_length_bp",
        "median_complete_length_bp",
    ):
        print(f"  {k}: {tr.get(k)}")

print("\nEnrichment (switch / background):")
for k in ("enrichment_frac_low_gq", "enrichment_frac_low_dp", "enrichment_frac_rnc_I", "enrichment_frac_sentinel"):
    print(f"  {k}: {summary.get(k)}")
nh = summary.get("nearest_het") or {}
if nh:
    print("\nNearest-het enrichment vs background hets:")
    print(f"  all: {nh.get('enrichment_frac_low_gq')}")
    print(f"  switch_at_hom_ref: {nh.get('hom_ref_switch_enrichment_frac_low_gq')}")
    print("  sides:", nh.get("side_counts"))
rnc = summary.get("rnc") or {}
if rnc:
    print("\nRNC-I (incomplete gVCF) vs background:")
    print(f"  enrichment: {rnc.get('enrichment_frac_I')}")
    print(f"  switch_at_hom_ref frac_I: {(rnc.get('switch_at_hom_ref') or {}).get('frac_I')}")
    print(f"  switch top codes: {(rnc.get('switch') or {}).get('top_codes')}")

HAS_GQ = summary["switch_site_calls"].get("median_gq") is not None
print("HAS_GQ:", HAS_GQ)

display(pd.DataFrame([
    {"set": "switch", **summary["switch_site_calls"]},
    {"set": "background", **summary["background_site_calls"]},
]))
display(pd.DataFrame([
    summary["switches_all"],
    summary["switches_flicker"],
    summary["switches_sustained"],
]))
if tr:
    display(pd.DataFrame([tr]))
switches.head()

## Are switches enriched for low GQ / DP?

If bad genotypes drive spurious LAI flips, switch sites should show **lower median GQ/DP**
and **higher fractions below threshold** than background, especially among **flickers**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def load_col(name: str) -> np.ndarray:
    p = RUN_DIR / name
    if not p.is_file():
        return np.array([])
    return pd.read_csv(p)["value"].to_numpy()

sw_gq = load_col("switch_gq.csv")
bg_gq = load_col("background_gq.csv")
sw_dp = load_col("switch_dp.csv")
bg_dp = load_col("background_dp.csv")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, sw, bg, thr, title in (
    (axes[0], sw_gq, bg_gq, GQ_THRESHOLD, "GQ"),
    (axes[1], sw_dp, bg_dp, DP_THRESHOLD, "DP"),
):
    if len(sw) == 0 or len(bg) == 0:
        ax.set_title(f"{title}: no data")
        continue
    hi = max(np.percentile(sw, 99), np.percentile(bg, 99), thr * 2)
    bins = np.linspace(0, hi, 40)
    ax.hist(bg, bins=bins, density=True, alpha=0.45, label="background")
    ax.hist(sw, bins=bins, density=True, alpha=0.55, label="switch")
    ax.axvline(thr, color="k", ls="--", lw=1, label=f"threshold={thr}")
    ax.set_xlabel(title)
    ax.set_ylabel("density")
    ax.legend(frameon=False)
    ax.set_title(title)

fig.suptitle(f"Quality at ancestry switches vs background ({PREFIX})")
fig.tight_layout()
fig_path = RUN_DIR / "gq_dp_switch_vs_bg.png"
fig.savefig(fig_path, dpi=150)
print("wrote", fig_path)
plt.show()

## Flicker vs sustained switches

Flickers (A→B→A) are the pattern that looks like a bad local genotype tract.
Sustained switches may be real ancestry breakpoints.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col, thr, title in (
    (axes[0], "gq", GQ_THRESHOLD, "GQ at switch"),
    (axes[1], "dp", DP_THRESHOLD, "DP at switch"),
):
    for label, mask in (
        ("flicker", switches["is_flicker"] == 1),
        ("sustained", switches["is_flicker"] == 0),
    ):
        vals = switches.loc[mask, col].dropna().to_numpy()
        if len(vals) == 0:
            continue
        hi = max(np.percentile(vals, 99), thr * 2)
        bins = np.linspace(0, hi, 30)
        ax.hist(vals, bins=bins, density=True, alpha=0.5, label=f"{label} (n={len(vals)})")
    ax.axvline(thr, color="k", ls="--", lw=1)
    ax.set_xlabel(col.upper())
    ax.set_title(title)
    ax.legend(frameon=False)

fig.tight_layout()
fig_path = RUN_DIR / "gq_dp_flicker_vs_sustained.png"
fig.savefig(fig_path, dpi=150)
print("wrote", fig_path)
plt.show()

print("flicker median GQ:", switches.loc[switches.is_flicker == 1, "gq"].median())
print("sustained median GQ:", switches.loc[switches.is_flicker == 0, "gq"].median())

## GQ / DP by genotype class

GLnexus often encodes cohort `0/0` as `GQ≤1, DP=0`. That sentinel dominates the pooled histograms. Split switch vs background **within** `hom_ref`, `het`, and `hom_alt`.

- If the zero spike is almost all `hom_ref` and het enrichment ≈ 1: the encoding is confounding QC, not FLARE.
- If **hets at switches** have lower GQ/DP than background hets: DeepVariant error is a plausible driver of flips.

`is_glnexus_homref_sentinel` is `hom_ref` and (`DP` is 0 or missing) and (`GQ` is missing or ≤1).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rows = []
for klass, block in (summary.get("by_gt") or {}).items():
    sw = block["switch"]
    bg = block["background"]
    rows.append({
        "gt_class": klass,
        "switch_n": sw["n"],
        "switch_frac": sw["frac_of_parent"],
        "switch_median_gq": sw["median_gq"],
        "switch_median_dp": sw["median_dp"],
        "switch_frac_low_gq": sw["frac_low_gq"],
        "switch_frac_sentinel": sw["frac_sentinel"],
        "bg_n": bg["n"],
        "bg_frac": bg["frac_of_parent"],
        "bg_median_gq": bg["median_gq"],
        "bg_median_dp": bg["median_dp"],
        "bg_frac_low_gq": bg["frac_low_gq"],
        "enrichment_low_gq": block["enrichment_frac_low_gq"],
        "enrichment_low_dp": block["enrichment_frac_low_dp"],
    })
by_gt = pd.DataFrame(rows)
display(by_gt)
by_gt.to_csv(RUN_DIR / "gq_dp_by_gt_summary.tsv", sep="\t", index=False)

print("het enrichment low-GQ:", (summary.get("by_gt") or {}).get("het", {}).get("enrichment_frac_low_gq"))
print("hom_ref sentinel frac (switch):", (summary.get("by_gt") or {}).get("hom_ref", {}).get("switch", {}).get("frac_sentinel"))

by_gt_vals = RUN_DIR / "gq_dp_by_gt.csv"
if by_gt_vals.is_file() and by_gt_vals.stat().st_size > 20:
    gt_vals = pd.read_csv(by_gt_vals)
    classes = ["hom_ref", "het", "hom_alt"]
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharey="row")
    for row_i, metric, thr in ((0, "gq", GQ_THRESHOLD), (1, "dp", DP_THRESHOLD)):
        for col_i, klass in enumerate(classes):
            ax = axes[row_i, col_i]
            sub = gt_vals[(gt_vals.gt_class == klass) & (gt_vals.metric == metric)]
            sw = sub.loc[sub.set == "switch", "value"].to_numpy()
            bg = sub.loc[sub.set == "background", "value"].to_numpy()
            if len(sw) == 0 and len(bg) == 0:
                ax.set_title(f"{klass} {metric}: no data")
                continue
            hi_candidates = [thr * 2]
            if len(sw):
                hi_candidates.append(float(np.percentile(sw, 99)))
            if len(bg):
                hi_candidates.append(float(np.percentile(bg, 99)))
            bins = np.linspace(0, max(hi_candidates), 30)
            if len(bg):
                ax.hist(bg, bins=bins, density=True, alpha=0.45, label=f"bg n={len(bg)}")
            if len(sw):
                ax.hist(sw, bins=bins, density=True, alpha=0.55, label=f"switch n={len(sw)}")
            ax.axvline(thr, color="k", ls="--", lw=1)
            ax.set_title(f"{klass} {metric.upper()}")
            ax.legend(frameon=False, fontsize=8)
            if col_i == 0:
                ax.set_ylabel("density")
    fig.suptitle(f"GQ/DP by GT class, switch vs background ({PREFIX})")
    fig.tight_layout()
    fig_path = RUN_DIR / "gq_dp_by_gt.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.show()
else:
    print("no gq_dp_by_gt.csv (re-run flare_switch_qc.py after staging the updated script)")

if "gt_class" in switches.columns:
    print("\nSwitch GT class counts:")
    print(switches["gt_class"].value_counts(dropna=False))
    if "is_glnexus_homref_sentinel" in switches.columns:
        print("sentinel switches:", int(switches["is_glnexus_homref_sentinel"].sum()))

## Nearest het GQ / DP at switches

FLARE does not use GQ/DP, and most switches land on `0/0` (GLnexus sentinel). The genotype *at* the AN change is often uninformative. This section looks at the **previous het** (last `0/1` or `1/0` before the switch) and the **nearest het** in bp.

- If the switch site is itself a het, nearest het = that site (`side=current`).
- If the switch is `hom_ref` / `hom_alt`, nearest het is the closer of previous and next het.

Compare those hets to **background hets**. If nearest-het GQ at `hom_ref` switches is like background, the HMM is not flipping because of a bad nearby het.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

nh = summary.get("nearest_het")
if not nh:
    print("summary.json has no nearest_het block. Restage scripts/ and re-run flare_switch_qc.py.")
else:
    rows = []
    for key in ("all", "flicker", "sustained", "prev_het_all", "switch_at_hom_ref", "switch_at_het"):
        block = nh.get(key) or {}
        rows.append({
            "set": key,
            "n": block.get("n"),
            "n_with_het": block.get("n_with_het"),
            "frac_with_het": block.get("frac_with_het"),
            "median_gq": block.get("median_gq"),
            "median_dp": block.get("median_dp"),
            "frac_gq_lt_thr": block.get("frac_gq_lt_thr"),
            "frac_dp_lt_thr": block.get("frac_dp_lt_thr"),
            "median_gap_bp": block.get("median_gap_bp"),
        })
    bg = nh.get("background_hets") or {}
    rows.append({
        "set": "background_hets",
        "n": bg.get("n"),
        "n_with_het": bg.get("n"),
        "frac_with_het": 1.0 if bg.get("n") else None,
        "median_gq": bg.get("median_gq"),
        "median_dp": bg.get("median_dp"),
        "frac_gq_lt_thr": bg.get("frac_low_gq"),
        "frac_dp_lt_thr": bg.get("frac_low_dp"),
        "median_gap_bp": None,
    })
    nh_tbl = pd.DataFrame(rows)
    display(nh_tbl)
    nh_tbl.to_csv(RUN_DIR / "nearest_het_summary.tsv", sep="\t", index=False)
    print("nearest-het enrichment low-GQ (all):", nh.get("enrichment_frac_low_gq"))
    print("nearest-het enrichment low-GQ (switch at hom_ref):", nh.get("hom_ref_switch_enrichment_frac_low_gq"))
    print("sides:", nh.get("side_counts"))

nh_csv = RUN_DIR / "nearest_het_gq_dp.csv"
gt_vals = RUN_DIR / "gq_dp_by_gt.csv"
if nh_csv.is_file() and nh_csv.stat().st_size > 20:
    near = pd.read_csv(nh_csv)
    bg_het_gq = np.array([])
    bg_het_dp = np.array([])
    if gt_vals.is_file():
        gtv = pd.read_csv(gt_vals)
        bg_het_gq = gtv.loc[(gtv.set == "background") & (gtv.gt_class == "het") & (gtv.metric == "gq"), "value"].to_numpy()
        bg_het_dp = gtv.loc[(gtv.set == "background") & (gtv.gt_class == "het") & (gtv.metric == "dp"), "value"].to_numpy()

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))

    def hist_cmp(ax, sw, bg, thr, title):
        if len(sw) == 0 and len(bg) == 0:
            ax.set_title(f"{title}: no data")
            return
        hi = [thr * 2]
        if len(sw):
            hi.append(float(np.percentile(sw, 99)))
        if len(bg):
            hi.append(float(np.percentile(bg, 99)))
        bins = np.linspace(0, max(hi), 30)
        if len(bg):
            ax.hist(bg, bins=bins, density=True, alpha=0.45, label=f"bg het n={len(bg)}")
        if len(sw):
            ax.hist(sw, bins=bins, density=True, alpha=0.55, label=f"nearest het n={len(sw)}")
        ax.axvline(thr, color="k", ls="--", lw=1)
        ax.set_title(title)
        ax.legend(frameon=False, fontsize=8)

    all_gq = near["gq"].dropna().to_numpy()
    all_dp = near["dp"].dropna().to_numpy()
    homref = near[near.gt_class == "hom_ref"]
    hr_gq = homref["gq"].dropna().to_numpy()
    hr_dp = homref["dp"].dropna().to_numpy()
    hist_cmp(axes[0, 0], all_gq, bg_het_gq, GQ_THRESHOLD, "Nearest-het GQ (all switches)")
    hist_cmp(axes[0, 1], all_dp, bg_het_dp, DP_THRESHOLD, "Nearest-het DP (all switches)")
    hist_cmp(axes[1, 0], hr_gq, bg_het_gq, GQ_THRESHOLD, "Nearest-het GQ (switch at hom_ref)")
    hist_cmp(axes[1, 1], hr_dp, bg_het_dp, DP_THRESHOLD, "Nearest-het DP (switch at hom_ref)")
    fig.suptitle(f"Flanking het quality at ancestry switches ({PREFIX})")
    fig.tight_layout()
    fig_path = RUN_DIR / "gq_dp_nearest_het.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, col, thr, title in (
        (axes[0], "gq", GQ_THRESHOLD, "Nearest-het GQ"),
        (axes[1], "dp", DP_THRESHOLD, "Nearest-het DP"),
    ):
        for label, mask in (
            ("flicker", near["switch_class"] == "flicker"),
            ("sustained", near["switch_class"] == "sustained"),
        ):
            vals = near.loc[mask, col].dropna().to_numpy()
            if len(vals) == 0:
                continue
            hi = max(float(np.percentile(vals, 99)), thr * 2)
            ax.hist(vals, bins=np.linspace(0, hi, 30), density=True, alpha=0.5, label=f"{label} n={len(vals)}")
        ax.axvline(thr, color="k", ls="--", lw=1)
        ax.set_xlabel(col.upper())
        ax.set_title(title)
        ax.legend(frameon=False)
    fig.tight_layout()
    fig_path = RUN_DIR / "gq_dp_nearest_het_flicker.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 4))
    gaps = near.loc[near.gt_class == "hom_ref", "gap_bp"].dropna().to_numpy()
    if len(gaps):
        hi = float(np.percentile(gaps, 99))
        ax.hist(gaps, bins=np.linspace(0, max(hi, 1), 40), density=True)
        ax.set_xlabel("bp to nearest het")
        ax.set_ylabel("density")
        ax.set_title(f"Gap to nearest het, switches at hom_ref (median={np.median(gaps):.0f})")
    else:
        ax.set_title("no hom_ref-switch nearest-het gaps")
    fig.tight_layout()
    fig_path = RUN_DIR / "nearest_het_gap.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.show()

    if "nearest_het_gq" in switches.columns:
        print("\nFrom switches.tsv.gz:")
        print("  prev_het median GQ:", switches["prev_het_gq"].median())
        print("  nearest_het median GQ:", switches["nearest_het_gq"].median())
        print("  nearest_het_side:\n", switches["nearest_het_side"].value_counts(dropna=False).to_string())
else:
    print("no nearest_het_gq_dp.csv (re-run flare_switch_qc.py after staging the updated script)")

## GLnexus RNC at switches

`RNC` is two characters (one per allele). `..` means GLnexus treated the genotype as called. `I` means incomplete gVCF coverage — the site that should often have been `./.` but was still emitted as `0/0`.

If `hom_ref` switches are enriched for `I` vs background `hom_ref`, FLARE is flipping at false reference alleles. If they are almost all `..`, the dummy GQ/DP is just GLnexus’s hom-ref placeholder and the breakpoint is positional or driven by nearby hets.

In [ ]:
import pandas as pd

rnc = summary.get("rnc")
if not rnc:
    print("summary.json has no rnc block. Restage scripts/ and re-run flare_switch_qc.py after AnnotateFlareGqDp copies RNC.")
else:
    rows = []
    for key in ("switch", "background", "switch_at_hom_ref", "switch_at_het", "nearest_het_all", "nearest_het_switch_at_hom_ref"):
        block = rnc.get(key) or {}
        rows.append({
            "set": key,
            "n": block.get("n"),
            "frac_I": block.get("frac_I"),
            "frac_clean": block.get("frac_clean"),
            "top_codes": block.get("top_codes"),
        })
    rnc_tbl = pd.DataFrame(rows)
    display(rnc_tbl)
    rnc_tbl.to_csv(RUN_DIR / "rnc_summary.tsv", sep="\t", index=False)
    print("RNC-I enrichment (all switches):", rnc.get("enrichment_frac_I"))
    print("RNC-I enrichment (hom_ref switches vs all bg):", rnc.get("hom_ref_switch_enrichment_frac_I"))
    het_rnc = (summary.get("by_gt") or {}).get("het", {})
    hom_rnc = (summary.get("by_gt") or {}).get("hom_ref", {})
    print("by_gt het enrichment_frac_rnc_I:", het_rnc.get("enrichment_frac_rnc_I"))
    print("by_gt hom_ref enrichment_frac_rnc_I:", hom_rnc.get("enrichment_frac_rnc_I"))

if "rnc" in switches.columns:
    print("\nSwitch-site RNC counts:")
    print(switches["rnc"].value_counts(dropna=False).head(12).to_string())
    hom = switches[switches.gt_class == "hom_ref"]
    if len(hom):
        print("\nhom_ref switch RNC:")
        print(hom["rnc"].value_counts(dropna=False).head(12).to_string())
        print("hom_ref switch frac I:", hom["rnc"].fillna("").str.contains("I").mean())
    if "nearest_het_rnc" in switches.columns:
        print("\nNearest-het RNC (hom_ref switches):")
        print(hom["nearest_het_rnc"].value_counts(dropna=False).head(12).to_string() if len(hom) else "n/a")
else:
    print("switches.tsv.gz has no rnc column (old scan)")

## Top samples / regions by switch density

In [ ]:
agg = {
    "n_switches": ("pos", "size"),
    "n_flicker": ("is_flicker", "sum"),
    "median_gq": ("gq", "median"),
    "median_dp": ("dp", "median"),
    "frac_low_gq": ("gq", lambda s: (s < GQ_THRESHOLD).mean()),
}
if "gt_class" in switches.columns:
    agg["frac_het"] = ("gt_class", lambda s: (s == "het").mean())
if "is_glnexus_homref_sentinel" in switches.columns:
    agg["frac_sentinel"] = ("is_glnexus_homref_sentinel", "mean")
if "nearest_het_gq" in switches.columns:
    agg["median_nearest_het_gq"] = ("nearest_het_gq", "median")
    agg["median_nearest_het_gap"] = ("nearest_het_gap", "median")

by_sample = switches.groupby("sample").agg(**agg).sort_values("n_switches", ascending=False)
display(by_sample.head(20))

switches = switches.copy()
switches["mb"] = switches["pos"] // 1_000_000
mb_agg = {
    "n_switches": ("pos", "size"),
    "n_flicker": ("is_flicker", "sum"),
    "median_gq": ("gq", "median"),
}
if "gt_class" in switches.columns:
    mb_agg["frac_het"] = ("gt_class", lambda s: (s == "het").mean())
by_mb = (
    switches.groupby(["chrom", "mb"])
    .agg(**mb_agg)
    .sort_values("n_switches", ascending=False)
)
display(by_mb.head(20))

by_sample.to_csv(RUN_DIR / "switches_by_sample.tsv", sep="\t")
by_mb.to_csv(RUN_DIR / "switches_by_mb.tsv", sep="\t")

## Old vs new FLARE (same samples, same region)

Skip this section until `NEW_VCF` is set. Use the 1 Mb `FlareByPopulation` merged anc VCF now; swap in full chr1 when training finishes.

Both runs use the **shared sample list** (new VCF IDs ∩ old VCF IDs, plus `ANALYSIS_SAMPLES` if set). That is the AFR/AMR-only, controls-dropped set if that is what you trained.

At 1 cM/Mb, expected switches/hap/Mb ≈ **T / 100**. Cohort-wide T ≈ 120 → ~1.2; AFR T ≈ 6 → ~0.06. A 1 Mb window censors tract lengths — compare rates first, then ECDFs on chr1.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd

assert shutil.which("bcftools"), "bcftools required on PATH"


def _localize(src: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if src.startswith("gs://"):
        if FORCE_RELOCALIZE or not dest.is_file():
            print(f"gsutil cp {src} {dest}")
            subprocess.check_call(["gsutil", "-m", "cp", src, str(dest)])
            if src.endswith((".vcf", ".vcf.gz", ".bcf", ".bcf.gz")):
                try:
                    subprocess.check_call(["gsutil", "-m", "cp", src + ".tbi", str(dest) + ".tbi"])
                except subprocess.CalledProcessError:
                    print(f"no .tbi for {src}; continuing")
        else:
            print(f"using cached {dest}")
        return dest
    path = Path(src)
    assert path.is_file(), path
    return path


def _vcf_samples(path: Path) -> list[str]:
    return [s for s in subprocess.check_output(["bcftools", "query", "-l", str(path)], text=True).splitlines() if s]


def _run_qc(vcf: Path, out_dir: Path, samples_path: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    cli = SCRIPTS / "flare_switch_qc.py"
    cli_src = cli.read_text()
    if "def query_format_string" not in cli_src or "AN-only FLARE VCFs OK" not in cli_src:
        raise RuntimeError(
            f"{cli} still queries FORMAT/GQ on AN-only FLARE VCFs. "
            'Run: gsutil -m rsync -r scripts/ "$WORKSPACE_BUCKET/scripts/" '
            "then re-run the first notebook cell before this one."
        )
    summary_path = out_dir / "summary.json"
    if summary_path.is_file() and not FORCE_COMPARE:
        try:
            prev = json.loads(summary_path.read_text())
        except json.JSONDecodeError:
            prev = {}
        if prev.get("tracts"):
            print(f"reuse {summary_path}")
            return
        print(f"re-scan {out_dir}: previous summary has no tract stats")
    cmd = [
        sys.executable,
        str(SCRIPTS / "flare_switch_qc.py"),
        "--vcf", str(vcf),
        "--out-dir", str(out_dir),
        "--samples", str(samples_path),
        "--gq-threshold", str(GQ_THRESHOLD),
        "--dp-threshold", str(DP_THRESHOLD),
        "--flicker-max-bp", str(FLICKER_MAX_BP),
        "--bg-keep-every", str(BG_KEEP_EVERY),
    ]
    if COMPARE_REGION:
        cmd.extend(["--region", COMPARE_REGION])
    if MAX_SITES is not None:
        cmd.extend(["--max-sites", str(MAX_SITES)])
    print(" ".join(cmd))
    subprocess.check_call(cmd)


COMPARE_OK = bool(NEW_VCF)
if not COMPARE_OK:
    print("NEW_VCF is empty — skip old-vs-new. Set NEW_FLARE_VCF when the new anc VCF is ready.")
else:
    cmp_dir = RUN_DIR / "compare"
    cmp_dir.mkdir(parents=True, exist_ok=True)
    old_vcf = vcf_path
    new_vcf = _localize(NEW_VCF, cmp_dir / Path(NEW_VCF).name)
    models_path = None
    if NEW_MODELS:
        models_path = _localize(NEW_MODELS, cmp_dir / Path(NEW_MODELS).name)

    extra = []
    if SAMPLES:
        samp = Path(SAMPLES)
        if str(SAMPLES).startswith("gs://"):
            samp = RUN_DIR / "samples.txt"
            if not samp.is_file():
                subprocess.check_call(["gsutil", "cp", SAMPLES, str(samp)])
        extra = [ln.split()[0] for ln in samp.read_text().splitlines() if ln.strip() and not ln.startswith("#")]
        extra_set = set(extra)
    else:
        extra_set = None

    shared = sorted(set(_vcf_samples(old_vcf)) & set(_vcf_samples(new_vcf)))
    if extra_set is not None:
        shared = [s for s in shared if s in extra_set]
    assert shared, "no shared samples between old and new FLARE VCFs"
    shared_path = cmp_dir / "shared_samples.txt"
    shared_path.write_text("\n".join(shared) + "\n")
    print(f"shared samples: {len(shared)}")

    old_dir = cmp_dir / "old"
    new_dir = cmp_dir / "new"
    _run_qc(old_vcf, old_dir, shared_path)
    _run_qc(new_vcf, new_dir, shared_path)

    old_sum = json.loads((old_dir / "summary.json").read_text())
    new_sum = json.loads((new_dir / "summary.json").read_text())
    old_sw = pd.read_csv(old_dir / "switches.tsv.gz", sep="\t", compression="gzip")
    new_sw = pd.read_csv(new_dir / "switches.tsv.gz", sep="\t", compression="gzip")
    print("old switches", old_sum["n_switches"], "new switches", new_sum["n_switches"])

In [ ]:
if not COMPARE_OK:
    print("skip")
else:
    from flare_switch_qc import expected_mean_tract_mb, expected_switches_per_hap_per_mb

    def _tract_row(label, summary, t_gen=None):
        tr = summary.get("tracts") or {}
        rate = tr.get("switches_per_hap_per_mb")
        exp = expected_switches_per_hap_per_mb(t_gen) if t_gen else None
        return {
            "run": label,
            "n_samples": summary.get("n_samples"),
            "n_switches": summary.get("n_switches"),
            "n_flicker": summary.get("n_flicker_switches"),
            "flicker_frac": tr.get("flicker_frac"),
            "span_mb": tr.get("span_mb"),
            "switches_per_hap_per_mb": rate,
            "frac_haps_with_switch": tr.get("frac_haps_with_switch"),
            "median_tract_mb": None if tr.get("median_length_bp") is None else tr["median_length_bp"] / 1e6,
            "median_complete_tract_mb": None
            if tr.get("median_complete_length_bp") is None
            else tr["median_complete_length_bp"] / 1e6,
            "implied_T": tr.get("implied_t_gen"),
            "model_T": t_gen,
            "expected_switches_per_hap_per_mb": exp,
            "expected_mean_tract_mb": expected_mean_tract_mb(t_gen) if t_gen else None,
        }

    models = None
    if models_path is not None:
        models = pd.read_csv(models_path, sep="\t")
        print("new models:")
        display(models)

    t_by_pop = {}
    if models is not None and "population" in models.columns and "t_gen" in models.columns:
        t_by_pop = dict(zip(models["population"].astype(str), models["t_gen"].astype(float)))

    cmp_tbl = pd.DataFrame([
        _tract_row("old (cohort T)", old_sum, OLD_FLARE_T),
        _tract_row("new (per-pop)", new_sum, None),
    ])
    display(cmp_tbl)
    cmp_tbl.to_csv(cmp_dir / "old_vs_new_summary.tsv", sep="\t", index=False)

    pop_map = None
    cov_path = Path(COVARIATES) if COVARIATES else None
    if cov_path and cov_path.is_file():
        usecols = ["research_id", "population"]
        pop_map = pd.read_csv(cov_path, usecols=usecols, dtype=str)
        pop_map["research_id"] = pop_map["research_id"].astype(str)
        print("covariates populations:", sorted(pop_map["population"].dropna().unique()))
    elif COVARIATES:
        print("COVARIATES not found:", COVARIATES)

    def _by_pop(switches, samples, span_mb, pop_map):
        samp = pd.DataFrame({"sample": samples})
        if pop_map is not None:
            samp = samp.merge(pop_map, left_on="sample", right_on="research_id", how="left")
        else:
            samp["population"] = "all"
        n_samp = samp.groupby(samp["population"].fillna("NA"))["sample"].nunique()
        sw = switches.copy()
        sw["sample"] = sw["sample"].astype(str)
        if pop_map is not None:
            sw = sw.merge(pop_map, left_on="sample", right_on="research_id", how="left")
        else:
            sw["population"] = "all"
        sw["population"] = sw["population"].fillna("NA")
        n_sw = sw.groupby("population").size()
        n_fl = sw.groupby("population")["is_flicker"].sum() if "is_flicker" in sw.columns else 0
        out = pd.DataFrame({"n_samples": n_samp, "n_switches": n_sw, "n_flicker": n_fl}).fillna(0)
        out["n_haps"] = out["n_samples"] * 2
        out["switches_per_hap_per_mb"] = out["n_switches"] / out["n_haps"] / span_mb
        out["flicker_frac"] = out["n_flicker"] / out["n_switches"].replace(0, pd.NA)
        return out

    span_old = (old_sum.get("tracts") or {}).get("span_mb") or 1.0
    span_new = (new_sum.get("tracts") or {}).get("span_mb") or span_old
    by_old = _by_pop(old_sw, shared, span_old, pop_map).assign(run="old")
    by_new = _by_pop(new_sw, shared, span_new, pop_map).assign(run="new")
    # New-run T from models; old-run T is the cohort-wide value for every pop.
    by_old["model_T"] = OLD_FLARE_T
    by_new["model_T"] = by_new.index.map(lambda p: t_by_pop.get(str(p)))
    by_old["expected_rate"] = by_old["model_T"].map(expected_switches_per_hap_per_mb)
    by_new["expected_rate"] = by_new["model_T"].map(
        lambda t: expected_switches_per_hap_per_mb(t) if pd.notna(t) else None
    )
    by_pop = pd.concat([by_old, by_new])
    display(by_pop)
    by_pop.to_csv(cmp_dir / "old_vs_new_by_pop.tsv", sep="\t")

In [ ]:
if not COMPARE_OK:
    print("skip")
else:
    import matplotlib.pyplot as plt
    import numpy as np

    def _ecdf(xs):
        xs = np.asarray(xs, dtype=float)
        xs = xs[np.isfinite(xs) & (xs > 0)]
        if xs.size == 0:
            return np.array([]), np.array([])
        xs = np.sort(xs)
        y = np.arange(1, xs.size + 1) / xs.size
        return xs, y

    def _load_lengths(run_dir: Path, complete=False) -> np.ndarray:
        name = "complete_tract_length_bp.csv" if complete else "tract_length_bp.csv"
        p = run_dir / name
        if not p.is_file():
            return np.array([])
        s = pd.read_csv(p)["value"]
        return s.to_numpy(dtype=float)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

    rate_df = by_pop.reset_index()
    if "population" not in rate_df.columns:
        rate_df = rate_df.rename(columns={"index": "population"})
    pops = list(rate_df["population"].unique())
    x = np.arange(len(pops))
    w = 0.35
    old_r = [
        float(rate_df[(rate_df.population == p) & (rate_df.run == "old")]["switches_per_hap_per_mb"].mean())
        if ((rate_df.population == p) & (rate_df.run == "old")).any()
        else np.nan
        for p in pops
    ]
    new_r = [
        float(rate_df[(rate_df.population == p) & (rate_df.run == "new")]["switches_per_hap_per_mb"].mean())
        if ((rate_df.population == p) & (rate_df.run == "new")).any()
        else np.nan
        for p in pops
    ]
    axes[0].bar(x - w / 2, old_r, w, label="old", color="#4C78A8")
    axes[0].bar(x + w / 2, new_r, w, label="new", color="#F58518")
    for i, p in enumerate(pops):
        exp_new = rate_df[(rate_df.population == p) & (rate_df.run == "new")]["expected_rate"]
        exp_old = rate_df[(rate_df.population == p) & (rate_df.run == "old")]["expected_rate"]
        if len(exp_old) and pd.notna(exp_old.iloc[0]):
            axes[0].plot([x[i] - w / 2, x[i] - w / 2], [exp_old.iloc[0], exp_old.iloc[0]], color="black", marker="_", markersize=12)
        if len(exp_new) and pd.notna(exp_new.iloc[0]):
            axes[0].plot([x[i] + w / 2, x[i] + w / 2], [exp_new.iloc[0], exp_new.iloc[0]], color="black", marker="_", markersize=12)
    axes[0].set_xticks(x, pops, rotation=30, ha="right")
    axes[0].set_ylabel("switches / hap / Mb")
    axes[0].set_title("Switch rate vs T/100 (black ticks)")
    axes[0].legend(frameon=False)

    fl_old = [
        float(rate_df[(rate_df.population == p) & (rate_df.run == "old")]["flicker_frac"].mean())
        if ((rate_df.population == p) & (rate_df.run == "old")).any()
        else np.nan
        for p in pops
    ]
    fl_new = [
        float(rate_df[(rate_df.population == p) & (rate_df.run == "new")]["flicker_frac"].mean())
        if ((rate_df.population == p) & (rate_df.run == "new")).any()
        else np.nan
        for p in pops
    ]
    axes[1].bar(x - w / 2, fl_old, w, label="old", color="#4C78A8")
    axes[1].bar(x + w / 2, fl_new, w, label="new", color="#F58518")
    axes[1].set_xticks(x, pops, rotation=30, ha="right")
    axes[1].set_ylabel("flicker fraction")
    axes[1].set_title("A→B→A within 50 kb")
    axes[1].set_ylim(0, 1)
    axes[1].legend(frameon=False)

    for label, rundir, color in (("old", old_dir, "#4C78A8"), ("new", new_dir, "#F58518")):
        xs, ys = _ecdf(_load_lengths(rundir) / 1e6)
        if xs.size:
            axes[2].plot(xs, ys, label=label, color=color)
    axes[2].set_xscale("log")
    axes[2].set_xlabel("tract length (Mb)")
    axes[2].set_ylabel("ECDF")
    axes[2].set_title("All tracts (window-censored)")
    axes[2].legend(frameon=False)

    fig.tight_layout()
    fig_path = cmp_dir / "old_vs_new_tracts.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.show()

    # Switch-count histogram per hap (old vs new)
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for label, sw, color in (("old", old_sw, "#4C78A8"), ("new", new_sw, "#F58518")):
        counts = sw.groupby(["sample", "hap"]).size()
        # include zero-switch haps
        n_haps = 2 * len(shared)
        zeros = n_haps - len(counts)
        vals = np.concatenate([counts.to_numpy(), np.zeros(max(zeros, 0), dtype=int)])
        if vals.size == 0:
            continue
        ax.hist(vals, bins=np.arange(0, int(vals.max()) + 2), alpha=0.45, label=label, color=color)
    ax.set_xlabel("switches per hap in window")
    ax.set_ylabel("haps")
    ax.set_title("Per-hap switch counts")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig_path = cmp_dir / "old_vs_new_hap_switches.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.show()

## Interpretation checklist

| Observation | Suggests |
|-------------|----------|
| Switch sites ≪ background median GQ/DP; enrichment ≫ 1 | Bad genotypes associated with flips |
| Flickers worse GQ/DP than sustained | Spurious local flips |
| High `frac_rnc_I` at switches | Incomplete gVCF (`RNC=I`) at flip loci; those `0/0`s should have been no-calls |
| `hom_ref` switches are almost all `RNC=..` | Dummy GQ/DP is GLnexus’s hom-ref placeholder, not a no-call recode |
| **Hets at switches** have lower GQ/DP than background hets | DeepVariant error can drive flips |
| **Nearest het** at `hom_ref` switches looks like background hets | Flip is not explained by a bad flanking het; look at FLARE/phasing or false `0/0` |
| Nearest-het GQ at `hom_ref` switches ≪ background hets | DeepVariant error at nearby hets is a plausible HMM driver |
| Switches clustered in telomeres / low-mappability | Hard regions, not necessarily GQ alone |
| Enrichment ≈ 1 even within hets | Look at phasing / FLARE params |
| New AFR/AMR switch rate ≪ old, near T/100 | Per-pop T is doing what we wanted |
| New flicker fraction still ~50% | Short A→B→A remains; T is not the whole story |
| 1 Mb median tract ≈ window length | Censoring; wait for chr1 for tract ECDFs |

Outputs in `RUN_DIR`:
- `switches.tsv.gz` — every switch event (`gt`, `rnc`, flanking het GQ/DP/RNC)
- `tracts.tsv.gz` — window-censored ancestry stretches
- `summary.json` — enrichment plus `tracts` (switches/hap/Mb, implied T)
- `gq_dp_*.png`, `gq_dp_by_gt.png`, `gq_dp_nearest_het.png`
- `compare/old_vs_new_summary.tsv`, `compare/old_vs_new_by_pop.tsv`, `compare/old_vs_new_tracts.png`